# 04.01 - Typed Cypher Queries

Orthograph provides a typed query contract for Cypher through `CypherReadQuery` and
`CypherWriteQuery`. These enforce at **class-definition time** that your Cypher is
syntactically valid and that every `$parameter` aligns with a declared `Params` field.

This notebook covers:
- Defining typed read and write queries (declarative style)
- How `cypher_template` is validated at definition time
- Running queries through a `CypherExecutor`
- Using `$param IS NULL OR` for optional filters
- The imperative escape hatch (and when you need it)

In [ ]:
from typing import Any, Optional

from pydantic import BaseModel

from orthograph import GraphDataModel, NodeModel, RelationshipModel
from orthograph.extensions.cypher import (
    CypherExecutor,
    CypherQueryDefinitionError,
    CypherReadQuery,
    CypherWriteQuery,
)

## 1. Define the domain model

We reuse the classic filmography domain: `Person`, `Movie`, and `ACTED_IN`.

In [ ]:
class Person(NodeModel):
    __label__ = "Person"
    __uid_field__ = "name"
    name: str
    born: Optional[int] = None


class Movie(NodeModel):
    __label__ = "Movie"
    __uid_field__ = "title"
    title: str
    released: int
    tagline: Optional[str] = None


class ActedIn(RelationshipModel):
    __label__ = "ACTED_IN"
    __source_type__ = Person
    __target_type__ = Movie
    role: str


model = GraphDataModel(
    name="Filmography",
    node_types=[Person, Movie],
    relationship_types=[ActedIn],
)
print("Model:", model.name)
print("Nodes:", model.node_labels)
print("Rels: ", model.relationship_labels)

## 2. Declarative read query

Set a `cypher_template` ClassVar with `$param` placeholders that match your `Params` model
fields. The base class:
- Parses the Cypher at definition time (catches syntax errors immediately)
- Checks that every `$param` corresponds to a `Params` field
- Provides a default `build()` that returns `(cypher_template, params.model_dump())`

You only implement `materialize()` — the per-record mapping from raw driver records to
your `Output` model.

In [ ]:
class MoviesByYearParams(BaseModel):
    released: int


class MoviesByYear(CypherReadQuery[MoviesByYearParams, Movie]):
    """Find all movies released in a given year."""

    Params = MoviesByYearParams
    Output = Movie
    name = "movies_by_year"
    cypher_template = (
        "MATCH (m:Movie {released: $released}) RETURN m.title, m.released, m.tagline"
    )

    def materialize(self, raw: dict[str, Any]) -> Movie:
        return Movie(
            title=raw["m.title"],
            released=raw["m.released"],
            tagline=raw.get("m.tagline"),
        )


# The class was defined — that means cypher_template parsed and params aligned.
print("Query name:    ", MoviesByYear.name)
print("Backend:       ", MoviesByYear.backend)
print("Params fields: ", list(MoviesByYear.Params.model_fields.keys()))
print("Output fields: ", list(MoviesByYear.Output.model_fields.keys()))

In [ ]:
# Call build() -- pure, no session needed
query = MoviesByYear()
cypher, params = query.build(MoviesByYearParams(released=1999))
print("Cypher:", cypher)
print("Params:", params)

## 3. Declarative write query

Write queries work the same way. Implement `interpret_result()` instead of `materialize()`.

In [ ]:
class CreateMovieParams(BaseModel):
    title: str
    released: int


class CreateMovie(CypherWriteQuery[CreateMovieParams, int]):
    """Create a new Movie node."""

    Params = CreateMovieParams
    name = "create_movie"
    cypher_template = "CREATE (m:Movie {title: $title, released: $released}) RETURN m"

    def interpret_result(self, raw: Any) -> int:
        # In a real driver, raw is a Result object with counters.
        return 1  # nodes created


query = CreateMovie()
cypher, params = query.build(CreateMovieParams(title="Speed", released=1994))
print("Cypher:", cypher)
print("Params:", params)

## 4. Running queries through CypherExecutor

The `CypherExecutor` is the single I/O seam. It:
1. Validates params via `Params.model_validate()` (rejects bad types before any DB call)
2. Calls `build()` (pure — no session)
3. Parses the produced Cypher (runtime syntax check — critical for imperative queries)
4. Opens a session and executes
5. Materializes each record via `materialize()` (reads) or `interpret_result()` (writes)

Below we use a `FakeGraphSession` to demonstrate without a live database.

In [ ]:
class FakeGraphSession:
    """Minimal stand-in for a graph driver session."""

    def __init__(self, records: list[dict[str, Any]]):
        self._records = records

    def __enter__(self):
        return self

    def __exit__(self, *exc):
        pass

    def run(self, cypher: str, **params: Any):
        print(f"  [FakeSession] RUN: {cypher}")
        print(f"  [FakeSession] WITH: {params}")
        return self._records


# Simulate two movies returned from the database
fake_records = [
    {
        "m.title": "The Matrix",
        "m.released": 1999,
        "m.tagline": "Welcome to the Real World",
    },
    {"m.title": "Fight Club", "m.released": 1999, "m.tagline": None},
]

executor = CypherExecutor(lambda: FakeGraphSession(fake_records))
results = executor.read(MoviesByYear(), {"released": 1999})

print(f"\nReturned {len(results)} Movie objects:")
for movie in results:
    print(f"  {movie.title} ({movie.released}) — {movie.tagline}")

## 5. Definition-time validation

If your `cypher_template` contains a `$param` that doesn't match a `Params` field, or
if the Cypher syntax is invalid, you get a `CypherQueryDefinitionError` **at class
definition time** — before any query ever runs. It inherits from `TypeError` so it is
also caught by generic `except TypeError` handlers.

In [ ]:
# Example 1: $param not declared on Params model
try:

    class BadParamQuery(CypherReadQuery[MoviesByYearParams, Movie]):
        Params = MoviesByYearParams
        Output = Movie
        name = "bad_param"
        cypher_template = (
            "MATCH (m:Movie {year: $year}) RETURN m"  # $year not in Params!
        )

        def materialize(self, raw):
            return Movie(**raw)
except CypherQueryDefinitionError as e:
    print(f"Caught at definition time: {e}")

In [ ]:
# Example 2: Invalid Cypher syntax
try:

    class BadSyntaxQuery(CypherReadQuery[MoviesByYearParams, Movie]):
        Params = MoviesByYearParams
        Output = Movie
        name = "bad_syntax"
        cypher_template = "MATSCH (m:Movie) RETRN m"  # typos!

        def materialize(self, raw):
            return Movie(**raw)
except CypherQueryDefinitionError as e:
    print(f"Caught at definition time: {e}")

## 6. Optional filters with `$param IS NULL OR`

For queries with optional parameters, use the Cypher pattern
`$param IS NULL OR n.prop = $param`. This keeps the query **static** (a single
`cypher_template` string), preserves definition-time validation, and lets you
pass `None` to skip a filter at runtime.

In [ ]:
class MovieFilterParams(BaseModel):
    released: Optional[int] = None
    title: Optional[str] = None


class MoviesFiltered(CypherReadQuery[MovieFilterParams, Movie]):
    """Find movies with optional year and title filters."""

    Params = MovieFilterParams
    Output = Movie
    name = "movies_filtered"
    cypher_template = (
        "MATCH (m:Movie) "
        "WHERE ($released IS NULL OR m.released = $released) "
        "  AND ($title IS NULL OR m.title = $title) "
        "RETURN m.title, m.released, m.tagline"
    )

    def materialize(self, raw: dict[str, Any]) -> Movie:
        return Movie(
            title=raw["m.title"],
            released=raw["m.released"],
            tagline=raw.get("m.tagline"),
        )


# build() with partial params (title=None means "don't filter by title")
query = MoviesFiltered()
cypher, params = query.build(MovieFilterParams(released=1999))
print("Cypher:", cypher)
print("Params:", params)
print()

# build() with no filters at all
cypher, params = query.build(MovieFilterParams())
print("Cypher:", cypher)
print("Params:", params)

## 7. The imperative escape hatch

For the rare case where the query **shape** genuinely changes at runtime —
e.g. conditionally adding `OPTIONAL MATCH` clauses, choosing different relationship
types, or varying `RETURN` columns — you can override `build()` directly and skip
the `cypher_template`.

**Trade-offs:**
- No definition-time validation (syntax is only checked at runtime by the executor)
- A `UserWarning` is emitted at class-definition time to surface this
- Cannot be introspected by `validate_cypher()` or the catalogue's `describe()`

Prefer the `$param IS NULL OR` pattern whenever possible.

In [ ]:
import warnings

from orthograph.extensions.cypher.base_models import CypherQuery


class ActorFilmographyParams(BaseModel):
    name: str
    include_directed: bool = False


# This will emit a UserWarning — that's expected and intentional.
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)

    class ActorFilmography(CypherReadQuery[ActorFilmographyParams, Movie]):
        """Find movies for an actor, optionally including directed films.

        This query's shape changes based on include_directed — it adds
        an OPTIONAL MATCH clause conditionally. That structural change
        requires the imperative style.
        """

        Params = ActorFilmographyParams
        Output = Movie
        name = "actor_filmography"

        def build(self, params: ActorFilmographyParams) -> CypherQuery:
            q = "MATCH (p:Person {name: $name})-[:ACTED_IN]->(m:Movie) "
            if params.include_directed:
                q += "OPTIONAL MATCH (p)-[:DIRECTED]->(m) "
            q += "RETURN m.title, m.released"
            return q, {"name": params.name}

        def materialize(self, raw: dict[str, Any]) -> Movie:
            return Movie(title=raw["m.title"], released=raw["m.released"])


# Without directed
query = ActorFilmography()
cypher, params = query.build(
    ActorFilmographyParams(name="Keanu Reeves", include_directed=False)
)
print("Without directed:")
print("  Cypher:", cypher)
print("  Params:", params)
print()

# With directed
cypher, params = query.build(
    ActorFilmographyParams(name="Keanu Reeves", include_directed=True)
)
print("With directed:")
print("  Cypher:", cypher)
print("  Params:", params)

## Summary

| Style | Set `cypher_template`? | Override `build()`? | Definition-time validation? | When to use |
|-------|:---------------------:|:-------------------:|:---------------------------:|-------------|
| **Declarative** | Yes | No (free default) | Yes | 90% of queries — fixed shape |
| **Imperative** | No | Yes | No (runtime only) | Structural shape changes |

**Guidelines:**
- Use `$param IS NULL OR n.prop = $param` for optional filters — keeps queries declarative.
- Use the imperative escape hatch only when the query *topology* changes (conditional
  `MATCH`/`OPTIONAL MATCH`, dynamic relationship types, variable `RETURN` columns).
- The executor always runs `parse_cypher()` on the produced string, so even imperative
  queries get a syntax check before hitting the database.